# Week 2 -- Function 2: Bayesian Optimisation (2D)

In [ ]:
# Cell 2: Imports
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

In [ ]:
# -- WEEKLY OBSERVATIONS LOG --------------------------------------------------------------------------------
# Each week: append one new tuple  ([x1, x2, ...], y_value)  to weekly_log.
# This cell is idempotent -- safe to re-run; it never double-appends.

import numpy as np

INITIAL_SIZE = 10   # number of samples in the original course-provided data
DATA_PATH_X  = '../data/function_2/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_2/initial_outputs.npy'

weekly_log = [
    ([0.0, 1.0], 0.02486946982716038),   # Week 1: submitted [0.000000, 1.000000], received 0.02486946982716038
    # Week 2: add next week's result here as ([x1, x2, ...], y_value)
]

# -- auto-sync to .npy (idempotent) ------------------------------------------------------------------
X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f"Synced {n_missing} new observation(s).")
else:
    print("Already up-to-date.")

print(f"X: {X_disk.shape} | Y: {Y_disk.shape}")
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f"Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)")


In [ ]:
# Cell 3: Load data and inspect
# Function 2: Noisy black-box ML model (log-likelihood scores), 2D input, Maximise

X = np.load('../data/function_2/initial_inputs.npy')
Y = np.load('../data/function_2/initial_outputs.npy')

print(f'Input  shape : {X.shape}   (n_samples x n_dimensions)')
print(f'Output shape : {Y.shape}  (n_samples,)')
print()

# Sort descending by Y value
X_list = list(X)
Y_list = list(Y)
pairs = sorted(zip(Y_list, X_list), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 62)
print('  All observations (sorted descending by Y)')
print('=' * 62)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    print(f'  [{i+1:2d}]  X = [{x_val[0]:.8f}, {x_val[1]:.8f}]   Y = {y_val:+.6f}{marker}')
print('=' * 62)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
print(f'\n  Best Y*  = {best_Y:.6f}')
print(f'  Best X*  = [{best_X[0]:.8f}, {best_X[1]:.8f}]')

In [ ]:
# Cell 4: Fit GP with log-transformed Y

# Log-transform to handle extreme scale differences across observations
Y_fit = np.log(np.abs(Y) + 1e-300)

# Fixed RBF kernel (course style -- no ConstantKernel, no optimisation)
kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
gp.fit(X, Y_fit)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                   : {gp.kernel_}')
print(f'  Log-marginal-likelihood  : {gp.log_marginal_likelihood_value_:.4f}')
print()

# Sanity check: predict at best known point
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
actual_log = np.log(np.abs(best_Y) + 1e-300)
print('  Sanity check at best known X*:')
print(f'    X*                     = [{best_X[0]:.6f}, {best_X[1]:.6f}]')
print(f'    GP predicted mean      = {pred_mean[0]:.4f}  (log-space)')
print(f'    Actual log(|Y*|)       = {actual_log:.4f}  (log-space)')
print(f'    GP predicted std       = {pred_std[0]:.8f}')
print('=' * 62)

In [ ]:
# Cell 5: UCB acquisition over 50x50 grid

# Build 50x50 meshgrid over [0, 1]^2
x1 = np.linspace(0, 1, 50)
x2 = np.linspace(0, 1, 50)
XX1, XX2 = np.meshgrid(x1, x2)
X_grid = np.column_stack([XX1.ravel(), XX2.ravel()])  # shape (2500, 2)

# GP predictions across the full grid
post_mean, post_std = gp.predict(X_grid, return_std=True)

# UCB acquisition: mean + beta * std
beta = 3.5  # Increased from 2.5: not improving, push exploration into new regions
acquisition = post_mean + beta * post_std  # shape (2500,)

# Next query = grid point with highest UCB
best_idx = np.argmax(acquisition)
next_x   = X_grid[best_idx]
next_acq = acquisition[best_idx]

# Portal submission string
portal_string = f'{next_x[0]:.6f}-{next_x[1]:.6f}'

print('=' * 62)
print('  UCB Acquisition  (beta = 3.5, 50x50 grid)')
print('=' * 62)
print(f'  Grid size            : {len(X_grid)} points (50x50)')
print(f'  Max UCB value        : {next_acq:.4f}')
print(f'  Next query point     : [{next_x[0]:.6f}, {next_x[1]:.6f}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 62)

In [ ]:
# Cell 7: Summary

print('=' * 62)
print('  SUMMARY -- Bayesian Optimisation Results')
print('=' * 62)
print(f'  Function             : 2D Noisy Black-Box ML Model (log-likelihood)')
print(f'  Objective            : Maximise')
print(f'  Kernel               : RBF(length_scale=0.1, fixed)')
print(f'  Acquisition function : UCB  (beta = 3.5)')
print(f'  Y transform          : log(|Y| + 1e-300)')
print(f'  Grid search          : 50x50 meshgrid (2,500 points)')
print()
print(f'  Current best X*      : [{best_X[0]:.6f}, {best_X[1]:.6f}]')
print(f'  Current best Y*      : {best_Y:.6f}')
print()
print(f'  Next query point     : [{next_x[0]:.6f}, {next_x[1]:.6f}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 62)

## Week 2 -- Reflection

**Function**: F2 -- Noisy Log-Likelihood Maximisation (2D)  
**Domain context**: Noisy output with multiple local optima

### Query
- **Method**: GP + UCB (beta=3.5), RBF kernel (length_scale=0.1, fixed)
- **Query type**: Exploration
- **Input submitted**: [1.000000, 0.326531]
- **Output received**: 0.14927 âœ“ IMPROVED
- **Best so far**: 0.14927

### What this result taught us
Improvement to y=0.149 at [1.0, 0.327]. High x1 region shows more promise than high x2. The function may have a peak somewhere in the high-x1 strip.

### Strategy for Week 3
Keep beta=3.5, explore mid-x2 range at high x1; the peak may be in the [1.0, 0.3-0.6] neighbourhood.